<a href="https://colab.research.google.com/github/miriamamin1213-ux/HPV-classification/blob/main/Hancock_GPT2_Narrative.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
# HANCOCK: frozen GPT-2 feature extraction and classifier-only training
# 1. Imports, reproducibility and configuration
import os, random, json, copy, gdown
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from transformers import GPT2Model, GPT2TokenizerFast
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

SEED = 42
BATCH_SIZE = 64
EPOCHS = 500
LEARNING_RATE = 1e-2
WEIGHT_DECAY = 1e-4
DROPOUT = 0.1
MODEL_NAME = "openai-community/gpt2"

def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# 2. LOAD AND MERGE HANCOCK CLINICAL/PATHOLOGICAL DATA
# ============================================================
CLINICAL_FILE = "clinical_data.json"
PATHOLOGICAL_FILE = "pathological_data.json"

if not os.path.exists(CLINICAL_FILE):
    if gdown.download(id="1F_kngwHn6KkOQqSwtYXCaF-bT8DZt__Q", output=CLINICAL_FILE, quiet=False) is None:
        raise FileNotFoundError("Could not download clinical_data.json.")

if not os.path.exists(PATHOLOGICAL_FILE):
    if gdown.download(id="1KY3NS9q3lfodf-t3g92S1JE3Vqcl6o3F", output=PATHOLOGICAL_FILE, quiet=False) is None:
        raise FileNotFoundError("Could not download pathological_data.json.")

def load_json_table(path):
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, list):
        return pd.DataFrame(obj)
    if isinstance(obj, dict):
        for key in ["data", "records", "patients", "items"]:
            if key in obj and isinstance(obj[key], list):
                return pd.DataFrame(obj[key])
        try:
            return pd.DataFrame(obj)
        except ValueError:
            pass
    raise ValueError(f"Could not convert {path} to a table.")

clinical_df = load_json_table(CLINICAL_FILE)
pathological_df = load_json_table(PATHOLOGICAL_FILE)
if ("patient_id" not in clinical_df.columns or "patient_id" not in pathological_df.columns):
    raise KeyError("Both JSON files must contain 'patient_id'.")

df = clinical_df.merge(pathological_df, on="patient_id", how="left", suffixes=("", "_path"))
print("Clinical shape:", clinical_df.shape)
print("Pathology shape:", pathological_df.shape)
print("Merged shape:", df.shape)
# ============================================================
# 3. SELECT SIX FEATURES AND DEFINE THE HPV TARGET
# ============================================================
def resolve_col(candidates, required=False):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"Required column not found: {candidates}. " f"Available columns: {df.columns.tolist()}")
    return None

target_col = resolve_col(["hpv_association_p16", "hpv_association_p16_path"], required=True)
feature_candidates = {
    "age_at_initial_diagnosis": ["age_at_initial_diagnosis", "age_at_initial_diagnosis_path"],
    "sex": ["sex", "sex_path"],
    "smoking_status": ["smoking_status", "smoking_status_path"],
    "primary_tumor_site": ["primary_tumor_site", "primary_tumor_site_path"],
    "primarily_metastasis": ["primarily_metastasis", "primarily_metastasis_path"],
    "pN_stage": ["pN_stage", "pN_stage_path", "pathological_N_stage", "pathological_n_stage"]
}
feature_map = {name: resolve_col(cols, required=True) for name, cols in feature_candidates.items()}
features = list(feature_candidates.keys())
df = df.dropna(subset=[target_col]).copy()

def normalize_target(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in ["negative", "neg", "0", "false", "no"]:
        return 0
    if s in ["positive", "pos", "1", "true", "yes"]:
        return 1
    try:
        n = float(s)
        if n in [0, 1]:
            return int(n)
    except ValueError:
        pass
    return np.nan

df["HPV_Label"] = df[target_col].apply(normalize_target)
df = df.dropna(subset=["HPV_Label"]).copy()
df["HPV_Label"] = df["HPV_Label"].astype(int)
feature_df = df[[feature_map[c] for c in features]].copy()
feature_df = feature_df.rename(columns={feature_map[c]: c for c in features})
df = pd.concat([feature_df, df[["patient_id", "HPV_Label"]]], axis=1)
# A row-level split is valid only if there is one eligible row per patient.
if df["patient_id"].isna().any() or df["patient_id"].duplicated().any():
    raise ValueError("Missing or duplicate patient IDs; resolve before patient-level splitting.")
df = df.loc[:, ~df.columns.duplicated()].copy()
df["age_at_initial_diagnosis"] = pd.to_numeric(df["age_at_initial_diagnosis"], errors="coerce")
print("Patients:", len(df))
print("HPV classes (0=negative, 1=positive):")
print(df["HPV_Label"].value_counts().sort_index())
print("Features:", features)
print("N-stage source:", feature_map["pN_stage"])
# ============================================================
# 4. STRATIFIED TRAIN/VALIDATION/TEST SPLIT (65/15/20)
# ============================================================
train_val_df, test_df = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df["HPV_Label"])
train_df, val_df = train_test_split(train_val_df, test_size=0.1875, random_state=SEED, stratify=train_val_df["HPV_Label"])
train_df, val_df, test_df = (split.copy() for split in [train_df, val_df, test_df])
# ============================================================
# 5. IMPUTE MISSING VALUES USING TRAINING DATA ONLY
# ============================================================
fill_values = {}
for c in features:
    if c == "age_at_initial_diagnosis":
        value = train_df[c].median()
        if pd.isna(value):
            value = 0
    else:
        # Preserve original category values, including numeric-coded categories.
        modes = train_df[c].dropna().mode()
        value = modes.iloc[0] if not modes.empty else "Unknown"
    fill_values[c] = value
    for split in [train_df, val_df, test_df]:
        split[c] = split[c].fillna(value)
        if split[c].dtype == object:
            split[c] = split[c].astype(str).replace({"": "Unknown", "nan": "Unknown", "None": "Unknown"})

y_train = train_df["HPV_Label"].to_numpy(dtype=int)
y_val = val_df["HPV_Label"].to_numpy(dtype=int)
y_test = test_df["HPV_Label"].to_numpy(dtype=int)
print("Train:", len(train_df), np.bincount(y_train, minlength=2))
print("Validation:", len(val_df), np.bincount(y_val, minlength=2))
print("Test:", len(test_df), np.bincount(y_test, minlength=2))

# Shared category vocabulary: unsupported values are unknown in every representation.
category_mapping = {
    "sex": {"female": 0, "male": 1},
    "smoking_status": {"non-smoker": 0, "former": 1, "smoker": 2},
    "primary_tumor_site": {"CUP": 0, "Oropharynx": 1},
    "primarily_metastasis": {"no": 0, "yes": 1},
    "pN_stage": {"pN0": 0, "pN1": 1, "pN2": 2, "pN2a": 3, "pN2b": 4,
                 "pN2c": 5, "pN3": 6, "pN3b": 7, "NX": 8}
}
for frame in (train_df, val_df, test_df):
    for col, mapping in category_mapping.items():
        frame[col] = frame[col].where(frame[col].isin(mapping), "Unknown")

# Initialise the same frozen backbone for each experiment.
gpt2 = GPT2Model.from_pretrained(MODEL_NAME).to(device)
gpt2.requires_grad_(False)
gpt2.eval()
hidden_size = gpt2.config.hidden_size

# 6. Clinical narrative representation
MODEL_LABEL = "GPT2_Narrative_V3_2_FeatureOrdering"


site_description = {
    "CUP": "The patient has carcinoma of unknown primary (CUP), with no identified primary tumour site.",
    "Oropharynx": "The primary tumour originates in the oropharynx."
}

metastasis_description = {
    "no": "No metastases were recorded at initial diagnosis.",
    "yes": "Metastases were present at initial diagnosis."
}

def patient_to_text(row):
    age = row["age_at_initial_diagnosis"]
    sex = row["sex"]
    smoking = row["smoking_status"]
    site = row["primary_tumor_site"]
    metastasis = row["primarily_metastasis"]
    stage = row["pN_stage"]

    if stage == "pN0":
        nodal = "No regional lymph node metastases were identified (pN0)."
    elif stage == "NX":
        nodal = "Regional lymph node involvement could not be assessed (NX)."
    elif stage in {"pN1", "pN2", "pN2a", "pN2b", "pN2c", "pN3", "pN3b"}:
        nodal = f"Regional lymph node metastases are present (pathological stage {stage})."
    else:
        nodal = f"The pathological regional lymph node stage is {stage}."

    narrative = (
        f"The patient is a {age}-year-old {sex} diagnosed with head and neck cancer. "
        f"{site_description[site]} "
        f"{metastasis_description[metastasis]} "
        f"{nodal} "
        f"{smoking_description[smoking]}"
    )

    question = (
        "\nQuestion: Based on the clinical findings, predict whether "
        "the tumour is HPV positive or HPV negative.\nAnswer:"
    )
    return narrative + question

X_train, X_val, X_test = [df.apply(patient_to_text, axis=1).tolist() for df in (train_df, val_df, test_df)]
print("Example text:", X_train[0])

# 7. Extract text embeddings
tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

@torch.no_grad()
def extract_embeddings(texts):
    embeddings = []
    for i in range(0, len(texts), BATCH_SIZE):
        inputs = tokenizer(texts[i:i+BATCH_SIZE], padding=True, return_tensors="pt").to(device)
        hidden = gpt2(**inputs, use_cache=False).last_hidden_state
        last_positions = inputs["attention_mask"].sum(dim=1) - 1
        pooled = hidden[torch.arange(hidden.size(0), device=device), last_positions]

        embeddings.append(pooled.cpu())
    return torch.cat(embeddings, dim=0)

X_train_emb, X_val_emb, X_test_emb = [extract_embeddings(X) for X in (X_train, X_val, X_test)]
print("Training embedding shape:", X_train_emb.shape)

# 8. Create DataLoaders from precomputed embeddings

def make_loader(X, y, shuffle=False):
    dataset = TensorDataset(X, torch.tensor(y, dtype=torch.long))
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, generator=torch.Generator().manual_seed(SEED))

train_loader = make_loader(X_train_emb, y_train, shuffle=True)
val_loader = make_loader(X_val_emb, y_val)
test_loader = make_loader(X_test_emb, y_test)

# 9. Initialise classifier

torch.manual_seed(SEED)
model = nn.Sequential(
    nn.LayerNorm(hidden_size),
    nn.Dropout(DROPOUT),
    nn.Linear(hidden_size, 128),
    nn.GELU(),
    nn.Dropout(DROPOUT),
    nn.Linear(128, 2)
).to(device)

class_counts = np.bincount(y_train, minlength=2)
assert np.all(class_counts > 0), "Both classes must be present in training data."
class_weights = torch.tensor(len(y_train) / (2 * class_counts), dtype=torch.float32, device=device)
print("Class weights:", class_weights)

def criterion(logits, labels):
    loss = nn.functional.cross_entropy(logits, labels, reduction="none")
    return (loss * class_weights[labels]).mean()

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))

# 10. Train and select best model

best_val_loss = float("inf")
best_weights = None
best_epoch = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(y)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            val_loss += criterion(model(X), y).item() * len(y)
    val_loss /= len(val_loader.dataset)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = copy.deepcopy(model.state_dict())
        best_epoch = epoch
    if epoch % 50 == 0:
        print(f"Epoch {epoch}/{EPOCHS}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, Best Epoch={best_epoch}")

model.load_state_dict(best_weights)
torch.save(best_weights, f"{MODEL_LABEL}_best_classifier.pt")
print(f"Best epoch: {best_epoch}, Best validation loss: {best_val_loss:.4f}")

# 11. Test evaluation

model.eval()
predictions, probabilities, labels = [], [], []
with torch.no_grad():
    for X, y in test_loader:
        logits = model(X.to(device))
        predictions.extend(logits.argmax(dim=1).cpu().numpy())
        probabilities.extend(logits.softmax(dim=1)[:, 1].cpu().numpy())
        labels.extend(y.numpy())

acc = accuracy_score(labels, predictions)
bal_acc = balanced_accuracy_score(labels, predictions)
auc = roc_auc_score(labels, probabilities)

print(f"Accuracy:          {acc:.4f}")
print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"ROC-AUC:           {auc:.4f}")


Device: cuda
Clinical shape: (763, 32)
Pathology shape: (763, 18)
Merged shape: (763, 49)
Patients: 332
HPV classes (0=negative, 1=positive):
HPV_Label
0    191
1    141
Name: count, dtype: int64
Features: ['age_at_initial_diagnosis', 'sex', 'smoking_status', 'primary_tumor_site', 'primarily_metastasis', 'pN_stage']
N-stage source: pN_stage
Train: 215 [123  92]
Validation: 50 [29 21]
Test: 67 [39 28]


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Example text: The patient is a 53-year-old male diagnosed with head and neck cancer. The primary tumour originates in the oropharynx. No metastases were recorded at initial diagnosis. No regional lymph node metastases were identified (pN0). The patient is a current tobacco smoker, with ongoing exposure and elevated tobacco-related cancer risk.
Question: Based on the clinical findings, predict whether the tumour is HPV positive or HPV negative.
Answer:
Training embedding shape: torch.Size([215, 768])
Class weights: tensor([0.8740, 1.1685], device='cuda:0')
Trainable parameters: 100226
Epoch 50/500: Train Loss=0.6845, Val Loss=0.6835, Best Epoch=40
Epoch 100/500: Train Loss=0.6187, Val Loss=0.6290, Best Epoch=95
Epoch 150/500: Train Loss=0.5980, Val Loss=0.6342, Best Epoch=116
Epoch 200/500: Train Loss=0.5883, Val Loss=0.6388, Best Epoch=116
Epoch 250/500: Train Loss=0.5821, Val Loss=0.6385, Best Epoch=116
Epoch 300/500: Train Loss=0.5983, Val Loss=0.6400, Best Epoch=116
Epoch 350/500: T